# 唐诗平仄二分类训练（论文实验版）

这个 Notebook 是在你原来的 `唐诗平仄二分类训练.ipynb` 基础上重做的增强版，重点解决了下面几个问题：

1. **改成读取你现在真正可用的重标注仿写文件**
   - `七言律诗_重标注.json`
   - `七言律绝_重标注.json`
   - `五言律绝_重标注.json`
   - `五言律诗_重标注.json`

2. **严格把 `○` 当成“不确定 / 不参与监督”的位置**
   - `平 -> 0`
   - `仄 -> 1`
   - `○ -> -100`（不参与 loss，不参与指标）

3. **增加论文实验设计**
   - **Exp-1：Target-only**  
     只用新标注仿写数据训练，测试也在仿写数据上。
   - **Exp-2：Original-only Zero-shot**  
     只用原始唐诗训练，直接测试到仿写数据上，作为跨域基线。
   - **Exp-3：Mixed Training**  
     原始唐诗 + 仿写训练集一起训练，再测试仿写测试集。
   - **Exp-4：Two-stage Transfer（推荐主实验）**  
     先用原始唐诗训练，再用仿写训练集继续微调，最后在仿写测试集评估。

4. **增加论文常用图表**
   - 数据集诗体分布图
   - 标签分布图
   - `loss-acc-f1` 训练曲线
   - 混淆矩阵
   - Precision / Recall / F1 报表
   - ROC 曲线、PR 曲线
   - 不同诗体的测试集表现对比
   - 错误案例导出

---

## 你现在应不应该直接用旧版 notebook 训练？

**不建议。**

因为旧版至少有这几个关键问题：

- 还在读取旧文件名 `*_仿写标注.json`，不是现在的 `*_重标注.json`
- 没有把 `○` 位置显式忽略
- 没有完整的论文实验设计
- 没有数据集诗体统计图
- 没有完整的训练可视化和误差分析

这个新版 notebook 已经把这些都补上了。

In [3]:
# Step 1. 安装 / 导入依赖
# 如果你的环境里还没有这些库，先取消注释运行安装行

# !pip install transformers datasets accelerate scikit-learn matplotlib pandas -q

import os
import re
import json
import math
import copy
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
)

from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.unicode_minus"] = False

NameError: name 'LRScheduler' is not defined

In [ ]:
# Step 2. 全局配置
# 这里统一控制路径、随机种子、模型、实验开关、训练超参数

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("D:\desktop\毕设\dataset")

# 优先使用本地模型目录；如果不存在，就退回 HuggingFace 名称
LOCAL_MODEL_DIR = DATA_DIR / "models" / "chinese-roberta-wwm-ext"
MODEL_NAME = str(LOCAL_MODEL_DIR) if LOCAL_MODEL_DIR.exists() else "hfl/chinese-roberta-wwm-ext"

OUTPUT_ROOT = DATA_DIR / "outputs_paper_pingze"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_PATTERN = "tang_poem_*.json"
TARGET_FILES = [
    DATA_DIR / "七言律诗_重标注.json",
    DATA_DIR / "七言律绝_重标注.json",
    DATA_DIR / "五言律绝_重标注.json",
    DATA_DIR / "五言律诗_重标注.json",
]

LABEL2ID = {"平": 0, "仄": 1}
ID2LABEL = {0: "平", 1: "仄"}
IGNORE_INDEX = -100

MAX_LENGTH = 32
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LOGGING_STEPS = 100

# 你可以根据显卡情况微调 epoch
EPOCHS_TARGET_ONLY = 5
EPOCHS_ORIGINAL_ONLY = 3
EPOCHS_MIXED = 4
EPOCHS_TRANSFER_STAGE1 = 3
EPOCHS_TRANSFER_STAGE2 = 4

RUN_EXPERIMENTS = {
    "target_only": True,
    "original_only_zero_shot": True,
    "mixed_training": True,
    "two_stage_transfer": True,
}

print("MODEL_NAME =", MODEL_NAME)
print("OUTPUT_ROOT =", OUTPUT_ROOT.resolve())
print("CUDA available =", torch.cuda.is_available())
print("Target files exist =", {p.name: p.exists() for p in TARGET_FILES})

In [ ]:
# Step 3. 基础工具函数
# - 文本清洗
# - 诗体识别
# - 绘图辅助

PUNCS = set("，。！？；：、,.!?;:；  \t\n")

def remove_punc(text: str) -> str:
    return "".join(ch for ch in str(text) if ch not in PUNCS)

def normalize_strain_text(text: str) -> str:
    text = remove_punc(str(text))
    text = text.replace("?", "○").replace("？", "○")
    return "".join(ch if ch in "平仄○" else "○" for ch in text)

def classify_poem_type(poem: dict) -> str:
    paragraphs = poem.get("paragraphs", [])
    clean_lines = [remove_punc(x) for x in paragraphs if remove_punc(x)]
    if not clean_lines:
        return "其他"
    lens = [len(x) for x in clean_lines]
    n = len(clean_lines)

    if all(x == 10 for x in lens):
        if n == 4:
            return "五言绝句"
        if n == 8:
            return "五言律诗"
    if all(x == 14 for x in lens):
        if n == 4:
            return "七言绝句"
        if n == 8:
            return "七言律诗"
    return "其他"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def show_bar(counter_like, title, xlabel="", ylabel="数量", rotate=0):
    items = list(counter_like.items())
    labels = [k for k, _ in items]
    values = [v for _, v in items]
    plt.figure(figsize=(9, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotate)
    for i, v in enumerate(values):
        plt.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 4. 浏览 dataset 中全部原始唐诗 json，并统计诗体分布
# 这一步就是你论文里“数据集描述图”的来源之一

base_files = sorted(DATA_DIR.glob(BASE_PATTERN))
print("当前扫描到的原始 json 文件数：", len(base_files))
print([p.name for p in base_files[:10]], "..." if len(base_files) > 10 else "")

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

original_poems = []
for p in base_files:
    original_poems.extend(load_json_list(p))

print("当前扫描到的原始唐诗总首数：", len(original_poems))

original_type_counter = Counter(classify_poem_type(poem) for poem in original_poems)
display(pd.DataFrame({
    "poem_type": list(original_type_counter.keys()),
    "count": list(original_type_counter.values())
}).sort_values("count", ascending=False))

show_bar(
    original_type_counter,
    title="原始唐诗数据集诗体分布",
    xlabel="诗体",
    ylabel="诗歌数量",
    rotate=0,
)

In [ ]:
# Step 5. 读取目标域：新标注仿写数据

target_poems = []
for p in TARGET_FILES:
    if p.exists():
        poems = load_json_list(p)
        for x in poems:
            x["source_domain"] = "target_aug"
            x["source_file"] = p.name
        target_poems.extend(poems)

print("目标域仿写诗总首数：", len(target_poems))
target_type_counter = Counter(classify_poem_type(poem) for poem in target_poems)

display(pd.DataFrame({
    "poem_type": list(target_type_counter.keys()),
    "count": list(target_type_counter.values())
}).sort_values("count", ascending=False))

show_bar(
    target_type_counter,
    title="新标注仿写数据诗体分布",
    xlabel="诗体",
    ylabel="诗歌数量",
    rotate=0,
)

In [ ]:
# Step 6. 把诗转成“逐行逐字平仄分类样本”
# 关键点：
# - 标点不参与训练
# - strain 中的“○”变成 -100，不参与监督
# - 同时保留 poem_id、poem_type、source_domain，便于后面做误差分析

for poem in original_poems:
    poem["source_domain"] = "original"
    poem["source_file"] = poem.get("source_file", "base_json")

def poems_to_line_samples(poems, domain_name):
    samples = []
    bad_cases = []

    for poem_idx, poem in enumerate(poems):
        poem_id = poem.get("id", f"{domain_name}_{poem_idx}")
        title = poem.get("title", "")
        author = poem.get("author", "")
        poem_type = classify_poem_type(poem)
        paragraphs = poem.get("paragraphs", [])
        strains = poem.get("strains", [])

        if len(paragraphs) != len(strains):
            bad_cases.append({
                "poem_id": poem_id,
                "title": title,
                "reason": "paragraphs/strains 行数不一致",
                "paragraphs_len": len(paragraphs),
                "strains_len": len(strains),
            })
            continue

        for line_idx, (line, strain) in enumerate(zip(paragraphs, strains)):
            clean_line = remove_punc(line)
            clean_strain = normalize_strain_text(strain)

            if len(clean_line) != len(clean_strain):
                bad_cases.append({
                    "poem_id": poem_id,
                    "title": title,
                    "line_idx": line_idx,
                    "reason": "去标点后字数与平仄长度不一致",
                    "line": line,
                    "clean_line": clean_line,
                    "strain": strain,
                    "clean_strain": clean_strain,
                    "line_len": len(clean_line),
                    "strain_len": len(clean_strain),
                })
                continue

            char_labels = []
            valid_count = 0
            circle_count = 0
            for ch in clean_strain:
                if ch == "平":
                    char_labels.append(0)
                    valid_count += 1
                elif ch == "仄":
                    char_labels.append(1)
                    valid_count += 1
                else:
                    char_labels.append(IGNORE_INDEX)
                    circle_count += 1

            # 一整句如果全是 ○，没有监督意义，直接跳过
            if valid_count == 0:
                continue

            samples.append({
                "poem_id": poem_id,
                "title": title,
                "author": author,
                "poem_type": poem_type,
                "source_domain": poem.get("source_domain", domain_name),
                "source_file": poem.get("source_file", ""),
                "line_idx": line_idx,
                "text": clean_line,
                "labels": char_labels,
                "valid_label_count": valid_count,
                "circle_count": circle_count,
            })

    return samples, bad_cases

original_samples, original_bad_cases = poems_to_line_samples(original_poems, "original")
target_samples, target_bad_cases = poems_to_line_samples(target_poems, "target_aug")

print("original_samples =", len(original_samples))
print("target_samples   =", len(target_samples))
print("original_bad_cases =", len(original_bad_cases))
print("target_bad_cases   =", len(target_bad_cases))

save_json(original_bad_cases[:200], OUTPUT_ROOT / "original_bad_cases_preview.json")
save_json(target_bad_cases[:200], OUTPUT_ROOT / "target_bad_cases_preview.json")

In [ ]:
# Step 7. 数据标签统计图
# 这里看三件事：
# 1) 平/仄数量
# 2) ○ 的数量
# 3) 不同诗体的逐行样本量

def count_labels(samples):
    c = Counter()
    for s in samples:
        for x in s["labels"]:
            if x == 0:
                c["平"] += 1
            elif x == 1:
                c["仄"] += 1
            elif x == IGNORE_INDEX:
                c["○"] += 1
    return c

original_label_counter = count_labels(original_samples)
target_label_counter = count_labels(target_samples)

print("原始数据标签统计：", original_label_counter)
print("仿写数据标签统计：", target_label_counter)

show_bar(original_label_counter, "原始唐诗：平/仄/○ 标签分布", ylabel="字级标签数量")
show_bar(target_label_counter, "仿写数据：平/仄/○ 标签分布", ylabel="字级标签数量")

show_bar(Counter(s["poem_type"] for s in target_samples), "仿写数据逐行样本的诗体分布", ylabel="行样本数")

In [ ]:
# Step 8. 划分目标域 train / valid / test（按 poem_id）
# 论文主评测统一在“目标域仿写数据”上完成
# 这样可以公平比较：
# - target-only
# - original-only zero-shot
# - mixed
# - two-stage transfer

target_poem_ids = sorted({s["poem_id"] for s in target_samples})

target_train_ids, target_temp_ids = train_test_split(
    target_poem_ids,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

target_valid_ids, target_test_ids = train_test_split(
    target_temp_ids,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
)

target_train_set = set(target_train_ids)
target_valid_set = set(target_valid_ids)
target_test_set = set(target_test_ids)

target_train_samples = [s for s in target_samples if s["poem_id"] in target_train_set]
target_valid_samples = [s for s in target_samples if s["poem_id"] in target_valid_set]
target_test_samples  = [s for s in target_samples if s["poem_id"] in target_test_set]

print("Target train lines =", len(target_train_samples))
print("Target valid lines =", len(target_valid_samples))
print("Target test  lines =", len(target_test_samples))

save_json({
    "target_train_poem_ids": target_train_ids,
    "target_valid_poem_ids": target_valid_ids,
    "target_test_poem_ids": target_test_ids,
}, OUTPUT_ROOT / "target_domain_split_ids.json")

In [ ]:
# Step 9. 构造 HuggingFace Dataset、Tokenizer、指标函数

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_datasetdict(train_samples, valid_samples, test_samples):
    return DatasetDict({
        "train": Dataset.from_list(train_samples),
        "validation": Dataset.from_list(valid_samples),
        "test": Dataset.from_list(test_samples),
    })

def tokenize_and_align_labels(batch):
    batch_chars = [list(text) for text in batch["text"]]
    tokenized = tokenizer(
        batch_chars,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for i, labels in enumerate(batch["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(IGNORE_INDEX)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(IGNORE_INDEX)
            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

def tokenize_dataset(dataset_dict):
    raw_columns = dataset_dict["train"].column_names
    return dataset_dict.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=raw_columns,
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true = []
    y_pred = []

    for pred_row, label_row in zip(preds, labels):
        for p, l in zip(pred_row, label_row):
            if l != IGNORE_INDEX:
                y_true.append(int(l))
                y_pred.append(int(p))

    acc = accuracy_score(y_true, y_pred)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_ze": p_bin,
        "recall_ze": r_bin,
        "f1_ze": f1_bin,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
    }

def build_model():
    return AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Tokenizer loaded:", tokenizer.__class__.__name__)

In [ ]:
# Step 10. 绘图与评估辅助函数
# 这一块会输出：
# - loss/accuracy/f1 曲线
# - 混淆矩阵
# - ROC 曲线
# - PR 曲线
# - 每类指标表
# - 按诗体分组的测试表现
# - 错误案例导出

def extract_eval_items(dataset_obj, prediction_output):
    logits = prediction_output.predictions
    labels = prediction_output.label_ids
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    y_true, y_pred, y_prob = [], [], []
    flat_rows = []

    for i in range(len(dataset_obj)):
        sample = dataset_obj[i]
        text = sample["text"]
        poem_type = sample["poem_type"]
        title = sample["title"]
        author = sample["author"]
        word_idx = 0

        for p, l, prob in zip(preds[i], labels[i], probs[i]):
            if l == IGNORE_INDEX:
                continue
            ch = text[word_idx] if word_idx < len(text) else ""
            y_true.append(int(l))
            y_pred.append(int(p))
            y_prob.append(float(prob[1]))
            flat_rows.append({
                "title": title,
                "author": author,
                "poem_type": poem_type,
                "char": ch,
                "true_label": ID2LABEL[int(l)],
                "pred_label": ID2LABEL[int(p)],
                "prob_ze": float(prob[1]),
                "is_correct": int(l == p),
            })
            word_idx += 1

    return np.array(y_true), np.array(y_pred), np.array(y_prob), pd.DataFrame(flat_rows)

def plot_log_history(trainer, exp_name):
    log_df = pd.DataFrame(trainer.state.log_history)
    if log_df.empty:
        print("没有可绘制的训练日志。")
        return log_df

    train_df = log_df[log_df["loss"].notna()] if "loss" in log_df.columns else pd.DataFrame()
    eval_df = log_df[log_df["eval_loss"].notna()] if "eval_loss" in log_df.columns else pd.DataFrame()

    # Loss 曲线
    plt.figure(figsize=(8, 5))
    if not train_df.empty and "step" in train_df.columns:
        plt.plot(train_df["step"], train_df["loss"], label="train_loss")
    if not eval_df.empty and "step" in eval_df.columns:
        plt.plot(eval_df["step"], eval_df["eval_loss"], label="eval_loss")
    plt.title(f"{exp_name} - Loss Curve")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Accuracy 曲线
    if not eval_df.empty and "eval_accuracy" in eval_df.columns:
        plt.figure(figsize=(8, 5))
        plt.plot(eval_df["step"], eval_df["eval_accuracy"], marker="o", label="eval_accuracy")
        plt.title(f"{exp_name} - Validation Accuracy")
        plt.xlabel("step")
        plt.ylabel("accuracy")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Macro-F1 曲线
    if not eval_df.empty and "eval_macro_f1" in eval_df.columns:
        plt.figure(figsize=(8, 5))
        plt.plot(eval_df["step"], eval_df["eval_macro_f1"], marker="o", label="eval_macro_f1")
        plt.title(f"{exp_name} - Validation Macro F1")
        plt.xlabel("step")
        plt.ylabel("macro_f1")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return log_df

def plot_confusion(y_true, y_pred, exp_name):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{exp_name} - Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["平", "仄"])
    plt.yticks(tick_marks, ["平", "仄"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.show()

def plot_roc_pr(y_true, y_prob, exp_name):
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob, pos_label=1)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"{exp_name} - ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # PR
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)
    pr_auc = auc(recall, precision)
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AUC = {pr_auc:.4f}")
    plt.title(f"{exp_name} - Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.show()

def evaluate_and_visualize(trainer, raw_test_samples, tokenized_test_dataset, exp_name, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    metrics = trainer.evaluate(tokenized_test_dataset, metric_key_prefix="test")
    print(f"\n===== {exp_name} Test Metrics =====")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    pred_output = trainer.predict(tokenized_test_dataset)
    y_true, y_pred, y_prob, flat_df = extract_eval_items(raw_test_samples, pred_output)

    log_df = plot_log_history(trainer, exp_name)
    plot_confusion(y_true, y_pred, exp_name)
    plot_roc_pr(y_true, y_prob, exp_name)

    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["平", "仄"],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).T
    display(report_df)

    # 按诗体统计准确率与 macro-f1
    grouped_rows = []
    for poem_type, subdf in flat_df.groupby("poem_type"):
        yt = subdf["true_label"].map({"平": 0, "仄": 1}).tolist()
        yp = subdf["pred_label"].map({"平": 0, "仄": 1}).tolist()
        acc = accuracy_score(yt, yp)
        _, _, f1_macro, _ = precision_recall_fscore_support(
            yt, yp, average="macro", zero_division=0
        )
        grouped_rows.append({
            "poem_type": poem_type,
            "token_count": len(subdf),
            "accuracy": acc,
            "macro_f1": f1_macro,
        })

    grouped_df = pd.DataFrame(grouped_rows).sort_values("token_count", ascending=False)
    display(grouped_df)

    if not grouped_df.empty:
        plt.figure(figsize=(8, 5))
        plt.bar(grouped_df["poem_type"], grouped_df["accuracy"])
        plt.title(f"{exp_name} - Test Accuracy by Poem Type")
        plt.ylabel("accuracy")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 5))
        plt.bar(grouped_df["poem_type"], grouped_df["macro_f1"])
        plt.title(f"{exp_name} - Test Macro F1 by Poem Type")
        plt.ylabel("macro_f1")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

    error_df = flat_df[flat_df["is_correct"] == 0].copy()
    error_df = error_df.sort_values(["prob_ze"], ascending=False)

    report_df.to_csv(save_dir / "classification_report.csv", encoding="utf-8-sig")
    grouped_df.to_csv(save_dir / "poem_type_metrics.csv", index=False, encoding="utf-8-sig")
    flat_df.to_csv(save_dir / "all_test_tokens.csv", index=False, encoding="utf-8-sig")
    error_df.to_csv(save_dir / "error_cases.csv", index=False, encoding="utf-8-sig")
    if not log_df.empty:
        log_df.to_csv(save_dir / "train_log_history.csv", index=False, encoding="utf-8-sig")
    save_json(metrics, save_dir / "test_metrics.json")

    return {
        "metrics": metrics,
        "report_df": report_df,
        "grouped_df": grouped_df,
        "errors_df": error_df,
        "flat_df": flat_df,
    }

In [ ]:
# Step 11. 通用训练函数
# 传入 train / valid / test 的样本列表，就能训练一个实验

def build_training_args(output_dir, num_epochs, metric_for_best_model="macro_f1"):
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epochs=num_epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=2,
    )

def train_one_stage_experiment(exp_name, train_samples, valid_samples, test_samples, num_epochs, init_model_dir=None):
    exp_dir = OUTPUT_ROOT / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    dataset = build_datasetdict(train_samples, valid_samples, test_samples)
    tokenized_dataset = tokenize_dataset(dataset)

    if init_model_dir is None:
        model = build_model()
    else:
        model = AutoModelForTokenClassification.from_pretrained(
            str(init_model_dir),
            num_labels=2,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
        )

    training_args = build_training_args(exp_dir, num_epochs=num_epochs)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    train_result = trainer.train()
    trainer.save_model(str(exp_dir / "best_model"))
    tokenizer.save_pretrained(str(exp_dir / "best_model"))

    eval_pack = evaluate_and_visualize(
        trainer=trainer,
        raw_test_samples=test_samples,
        tokenized_test_dataset=tokenized_dataset["test"],
        exp_name=exp_name,
        save_dir=exp_dir,
    )

    return {
        "trainer": trainer,
        "train_result": train_result,
        "eval_pack": eval_pack,
        "exp_dir": exp_dir,
    }

## 实验一：Target-only

只用**新标注仿写数据的训练集**训练，在**仿写测试集**上评估。

这个实验回答的问题是：

> 如果完全不借助原始唐诗，只靠仿写标注数据，模型能做到什么程度？

In [ ]:
# Step 12. Exp-1: Target-only

if RUN_EXPERIMENTS["target_only"]:
    target_only_result = train_one_stage_experiment(
        exp_name="exp1_target_only",
        train_samples=target_train_samples,
        valid_samples=target_valid_samples,
        test_samples=target_test_samples,
        num_epochs=EPOCHS_TARGET_ONLY,
        init_model_dir=None,
    )
else:
    print("已跳过 exp1_target_only")

## 实验二：Original-only Zero-shot

只用**原始唐诗数据**训练，然后**直接测试到仿写测试集**。

这个实验回答的问题是：

> 原始唐诗训练出的模型，跨域到仿写数据时，性能会掉多少？

这是很重要的**跨域基线**。

In [ ]:
# Step 13. Exp-2: Original-only Zero-shot
# 训练集/验证集都取原始唐诗；测试集固定为 target_test_samples
# 原始域内部切一小部分做验证，仅用于早停

if RUN_EXPERIMENTS["original_only_zero_shot"]:
    original_poem_ids = sorted({s["poem_id"] for s in original_samples})
    ori_train_ids, ori_valid_ids = train_test_split(
        original_poem_ids,
        test_size=0.1,
        random_state=SEED,
        shuffle=True,
    )
    ori_train_set = set(ori_train_ids)
    ori_valid_set = set(ori_valid_ids)

    original_train_samples = [s for s in original_samples if s["poem_id"] in ori_train_set]
    original_valid_samples = [s for s in original_samples if s["poem_id"] in ori_valid_set]

    original_only_result = train_one_stage_experiment(
        exp_name="exp2_original_only_zero_shot",
        train_samples=original_train_samples,
        valid_samples=original_valid_samples,
        test_samples=target_test_samples,   # 注意：测试域是 target
        num_epochs=EPOCHS_ORIGINAL_ONLY,
        init_model_dir=None,
    )
else:
    print("已跳过 exp2_original_only_zero_shot")

## 实验三：Mixed Training

把**原始唐诗训练样本**和**仿写训练集样本**混合训练，然后在**仿写测试集**上评估。

这个实验回答的问题是：

> 直接“全部混在一起训练”是否有效？

In [ ]:
# Step 14. Exp-3: Mixed Training

if RUN_EXPERIMENTS["mixed_training"]:
    mixed_train_samples = original_samples + target_train_samples
    mixed_valid_samples = target_valid_samples   # 验证仍然盯住目标域
    mixed_test_samples = target_test_samples

    mixed_result = train_one_stage_experiment(
        exp_name="exp3_mixed_training",
        train_samples=mixed_train_samples,
        valid_samples=mixed_valid_samples,
        test_samples=mixed_test_samples,
        num_epochs=EPOCHS_MIXED,
        init_model_dir=None,
    )
else:
    print("已跳过 exp3_mixed_training")

## 实验四：Two-stage Transfer（推荐主实验）

### Stage-1
先用**原始唐诗**训练，让模型先学到比较稳定的平仄知识。

### Stage-2
再用**仿写训练集**继续微调，让模型适应你的目标域标注风格。

这个实验通常最适合作为论文主实验，因为它同时兼顾了：

- 原始大数据带来的知识迁移
- 目标域小数据带来的风格适配

In [ ]:
# Step 15. Exp-4: Two-stage Transfer

if RUN_EXPERIMENTS["two_stage_transfer"]:
    transfer_stage1_dir = OUTPUT_ROOT / "exp4_transfer_stage1"
    transfer_stage1_dir.mkdir(parents=True, exist_ok=True)

    # Stage-1: original pretraining / finetuning
    original_poem_ids = sorted({s["poem_id"] for s in original_samples})
    ori_train_ids, ori_valid_ids = train_test_split(
        original_poem_ids,
        test_size=0.1,
        random_state=SEED,
        shuffle=True,
    )
    ori_train_set = set(ori_train_ids)
    ori_valid_set = set(ori_valid_ids)

    original_train_samples = [s for s in original_samples if s["poem_id"] in ori_train_set]
    original_valid_samples = [s for s in original_samples if s["poem_id"] in ori_valid_set]

    stage1_dataset = build_datasetdict(original_train_samples, original_valid_samples, target_test_samples)
    stage1_tokenized = tokenize_dataset(stage1_dataset)

    stage1_model = build_model()
    stage1_args = build_training_args(transfer_stage1_dir, num_epochs=EPOCHS_TRANSFER_STAGE1)

    stage1_trainer = Trainer(
        model=stage1_model,
        args=stage1_args,
        train_dataset=stage1_tokenized["train"],
        eval_dataset=stage1_tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    stage1_trainer.train()
    stage1_best_dir = transfer_stage1_dir / "best_model"
    stage1_trainer.save_model(str(stage1_best_dir))
    tokenizer.save_pretrained(str(stage1_best_dir))
    print("Transfer Stage-1 model saved to:", stage1_best_dir)

    # Stage-2: continue finetune on target train
    transfer_stage2_result = train_one_stage_experiment(
        exp_name="exp4_transfer_stage2",
        train_samples=target_train_samples,
        valid_samples=target_valid_samples,
        test_samples=target_test_samples,
        num_epochs=EPOCHS_TRANSFER_STAGE2,
        init_model_dir=stage1_best_dir,
    )
else:
    print("已跳过 exp4_transfer_stage2")

In [ ]:
# Step 16. 汇总所有实验结果，生成论文表格

summary_rows = []

def collect_summary(exp_name, result_obj):
    if result_obj is None:
        return
    metrics = result_obj["eval_pack"]["metrics"]
    summary_rows.append({
        "experiment": exp_name,
        "test_accuracy": metrics.get("test_accuracy"),
        "test_macro_f1": metrics.get("test_macro_f1"),
        "test_precision_ze": metrics.get("test_precision_ze"),
        "test_recall_ze": metrics.get("test_recall_ze"),
        "test_f1_ze": metrics.get("test_f1_ze"),
        "test_loss": metrics.get("test_loss"),
    })

if RUN_EXPERIMENTS["target_only"] and "target_only_result" in globals():
    collect_summary("Exp-1 Target-only", target_only_result)

if RUN_EXPERIMENTS["original_only_zero_shot"] and "original_only_result" in globals():
    collect_summary("Exp-2 Original-only Zero-shot", original_only_result)

if RUN_EXPERIMENTS["mixed_training"] and "mixed_result" in globals():
    collect_summary("Exp-3 Mixed Training", mixed_result)

if RUN_EXPERIMENTS["two_stage_transfer"] and "transfer_stage2_result" in globals():
    collect_summary("Exp-4 Two-stage Transfer", transfer_stage2_result)

summary_df = pd.DataFrame(summary_rows).sort_values("test_macro_f1", ascending=False)
display(summary_df)

summary_df.to_csv(OUTPUT_ROOT / "experiment_summary.csv", index=False, encoding="utf-8-sig")
print("实验汇总表已保存到：", OUTPUT_ROOT / "experiment_summary.csv")

## 论文里建议怎么写实验设计？

### 推荐主线
把 **Exp-4 Two-stage Transfer** 作为主方法，把另外三个作为对比基线：

- **Baseline-1：Target-only**
- **Baseline-2：Original-only Zero-shot**
- **Baseline-3：Mixed Training**
- **Ours / Main：Two-stage Transfer**

### 推荐论文章节解释逻辑
1. **Original-only**  
   说明“原始唐诗和仿写数据存在域差异”，直接迁移会掉点。
2. **Target-only**  
   说明“目标域小数据单独训练可行，但上限受限”。
3. **Mixed Training**  
   说明“简单混合训练有帮助，但不一定最优”。
4. **Two-stage Transfer**  
   说明“先学共性知识，再适配目标域”更合理。

### 不推荐作为主方法的方案
**先学仿写，再回到原始唐诗继续训练** 不建议做主实验。  
因为你的最终目标更像是**提升模型在新标注仿写数据上的效果**，不是反过来适配原始唐诗。

---

## 你在论文里可以直接写的结论方向

如果最后结果符合预期，通常可以这样表述：

- 原始唐诗数据提供了稳定的平仄知识先验；
- 新标注仿写数据提供了目标域风格信息；
- 两阶段迁移能更好地兼顾“通用知识”和“目标域适配”；
- 因而在目标域测试集上取得最佳 F1 / Accuracy。

---

## 当前这个工作区要特别注意的一点

我现在在当前环境里实际只扫描到了这些原始 JSON：

- `tang_poem_0.json`
- `tang_poem_1000.json`
- ...
- `tang_poem_20000.json`

也就是**当前目录里只有 21 个原始 JSON，共 21000 首诗**。  
所以本 notebook 现在统计出来的“原始数据诗体分布图”，反映的是**当前工作区可见的 21000 首**，不是你口头提到的全部 `tang_poem_0 ~ tang_poem_57000`。

如果你后面把其余 json 也放到同一目录，这个 notebook 会**自动重新扫描并更新统计图**。

In [ ]:
# Step 17. 训练后单句推理函数
# 可以快速看某一句的逐字预测结果

def predict_pingze(text, model_dir=None):
    model_dir = model_dir or (OUTPUT_ROOT / "exp4_transfer_stage2" / "best_model")
    infer_model = AutoModelForTokenClassification.from_pretrained(str(model_dir))
    infer_model.eval()

    clean_chars = [ch for ch in text if ch not in PUNCS]
    if len(clean_chars) == 0:
        return {"raw_text": text, "clean_text": "", "pred": ""}

    enc = tokenizer(
        [clean_chars],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    with torch.no_grad():
        outputs = infer_model(**enc)
        pred_ids = outputs.logits.argmax(dim=-1)[0].cpu().tolist()

    word_ids = enc.word_ids(batch_index=0)
    char_preds = []
    seen = set()
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in seen:
            continue
        seen.add(word_idx)
        char_preds.append(ID2LABEL[pred_ids[token_idx]])

    result = []
    j = 0
    for ch in text:
        if ch in PUNCS:
            result.append(ch)
        else:
            result.append(char_preds[j])
            j += 1

    return {
        "raw_text": text,
        "clean_text": "".join(clean_chars),
        "pred": "".join(result),
    }

# 示例（需要先完成训练）
# predict_pingze("春眠不觉晓，处处闻啼鸟。")